# Estrazione Codice Amministrativo da Normattiva

Estrae gli articoli della **L. 7 agosto 1990, n. 241** direttamente da Normattiva e genera il CSV `src/data/statutes/codice_amministrativo_normattiva.csv`.

Schema output:
- `numero`
- `titolo`
- `contenuto`
- `reference` (lista JSON di riferimenti interni al codice)
- `external_reference` (lista JSON di riferimenti esterni)

Integrazione Neo4j:
- `src/db/db_orchestrator.py` legge `reference` e `external_reference` in ingestione.
- I riferimenti interni in `reference` sono usati per creare relazioni `(:Statute)-[:CITES]->(:Statute)` intra-codice.

- I riferimenti sono normalizzati con correzioni di suffisso e whitelist interna: se un articolo non esiste nel codice, viene spostato in `external_reference`/`external_references`.


In [1]:
from __future__ import annotations

import csv
import json
import html
import http.cookiejar
import re
import urllib.request
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

BASE_NORMATTIVA_URL = "https://www.normattiva.it/uri-res/N2Ls?urn:nir:stato:legge:1990-08-07;241"
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_CSV_PATH = PROJECT_ROOT / "src/data/statutes/codice_amministrativo_normattiva.csv"
RAW_DEBUG_DIR = PROJECT_ROOT / "notebooks/tmp_normattiva_amministrativo"
SAVE_RAW_HTML = False
INCLUDE_UPDATES = False

ARTICLE_LINK_RE = re.compile(
    r"onclick=\"return showArticle\('(/atto/caricaArticolo\?[^']+)'\s*,\s*this\);\"[^>]*class=\"numero_articolo\"[^>]*>\s*([^<]+?)\s*</a>",
    re.IGNORECASE,
)
ART_REF_RE = re.compile(r"\bart(?:t|icolo)?\.?\s*(\d+(?:-[a-z]+)*(?:\.\d+)?)", re.IGNORECASE)
EXTERNAL_REFERENCE_MARKERS = (
    "c.p.",
    "codice penale",
    "c.c.",
    "codice civile",
    "c.p.p",
    "codice di procedura penale",
    "c.p.c",
    "codice di procedura civile",
    "cost.",
    "costituzione",
    "decreto",
    "d.lgs",
    "d.l.",
    "dpr",
)


@dataclass
class AdministrativeArticleEntry:
    numero: str
    titolo: str
    contenuto: str
    reference: str
    external_reference: str


def _build_opener() -> urllib.request.OpenerDirector:
    jar = http.cookiejar.CookieJar()
    opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(jar))
    opener.addheaders = [("User-Agent", "Mozilla/5.0")]
    return opener


def _normalize_article_label(raw_label: str) -> str:
    label = html.unescape(raw_label).strip().lower()
    label = label.replace("‑", "-").replace("–", "-").replace("—", "-")
    label = re.sub(r"\s+", "-", label)
    label = label.strip(". ")
    label = re.sub(r"^art\.?-?", "", label)
    return label


_SUFFIX_ORDER = {
    "bis": 1,
    "ter": 2,
    "quater": 3,
    "quinquies": 4,
    "sexies": 5,
    "septies": 6,
    "octies": 7,
    "novies": 8,
    "decies": 9,
    "undecies": 10,
    "duodecies": 11,
    "terdecies": 12,
    "quaterdecies": 13,
    "quinquiesdecies": 14,
    "sexiesdecies": 15,
    "septiesdecies": 16,
    "duodevicies": 17,
    "vicies": 18,
}


def _token_sort_key(token: str) -> tuple[int, int | str]:
    if token.isdigit():
        return (2, int(token))
    order = _SUFFIX_ORDER.get(token)
    if order is not None:
        return (1, order)
    return (3, token)


def _article_sort_key(label: str) -> tuple[int, tuple[tuple[int, int | str], ...], str]:
    normalized = _normalize_article_label(label)
    m = re.match(r"^(\d+)(.*)$", normalized)
    if not m:
        return (10**9, tuple(), normalized)
    base = int(m.group(1))
    rest = m.group(2).strip("-./")
    if not rest:
        return (base, tuple(), "")
    tokens = [t for t in re.split(r"[-./]", rest) if t]
    token_keys = tuple(_token_sort_key(t) for t in tokens)
    return (base, token_keys, rest)


def _html_to_lines(fragment: str) -> list[str]:
    text = re.sub(r"(?is)<script.*?>.*?</script>", " ", fragment)
    text = re.sub(r"(?is)<style.*?>.*?</style>", " ", text)
    text = re.sub(r"(?i)<br\s*/?>", "\n", text)
    text = re.sub(r"(?i)</div>|</p>|</li>|</h\d>", "\n", text)
    text = re.sub(r"(?is)<[^>]+>", " ", text)
    text = html.unescape(text)
    lines = [ln.strip() for ln in text.splitlines()]
    return [ln for ln in lines if ln]


def _clean_title(raw_title: str) -> str:
    title = raw_title.strip()
    title = re.sub(r"^[\(\)\.\s]+", "", title)
    title = re.sub(r"[\(\)\.\s]+$", "", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def _extract_title_and_body(lines: list[str], label: str) -> tuple[str, str]:
    if not lines:
        return "", ""

    heading_pattern = re.escape(label)
    heading_pattern = heading_pattern.replace("-", r"[-\s]?")
    art_re = re.compile(rf"^Art\.\s*{heading_pattern}\.?(?:\s|$)", re.IGNORECASE)

    art_idx = 0
    for i, line in enumerate(lines):
        if art_re.search(line):
            art_idx = i
            break

    content = lines[art_idx + 1 :] if art_idx + 1 < len(lines) else []
    if not content:
        return "", ""

    raw_title = content[0]
    title = _clean_title(raw_title)

    looks_like_body = (
        bool(re.match(r"^(Chiunque|Fuori|Quando|Qualora|La|Le|Il|I|Non|Nei|Nel|Se|Ove|Per)\b", title, re.IGNORECASE))
        and len(title) > 60
    )

    if looks_like_body:
        title = ""
        body_lines = content
    else:
        body_lines = content[1:]

    if not INCLUDE_UPDATES:
        cut_idx = None
        for i, ln in enumerate(body_lines):
            up = ln.upper()
            if up.startswith("AGGIORNAMENTO") or ln.startswith("------------"):
                cut_idx = i
                break
        if cut_idx is not None:
            body_lines = body_lines[:cut_idx]

    body = " ".join(body_lines).strip()
    body = re.sub(r"\s+", " ", body)
    body = re.sub(r"^\.\s*", "", body)
    return title, body


REF_SUFFIX_CORRECTIONS = {
    "nonies": "novies",
    "sexiesdecies": "sexdecies",
}


def _normalize_reference_suffixes(token: str) -> str:
    parts = token.split("-")
    if len(parts) <= 1:
        return token
    fixed = [parts[0]]
    for part in parts[1:]:
        fixed.append(REF_SUFFIX_CORRECTIONS.get(part, part))
    return "-".join(fixed)


def _normalize_reference_article(raw_ref: str) -> str:
    token = raw_ref.strip().lower()
    token = token.replace("‑", "-").replace("–", "-").replace("—", "-")
    m = re.search(r"(\d+(?:-[a-z]+)*(?:\.\d+)?)", token)
    if not m:
        return ""
    return _normalize_reference_suffixes(m.group(1))


def _extract_references(text: str, valid_internal_refs: set[str] | None = None) -> tuple[list[str], list[str]]:
    if not text:
        return [], []

    internal: set[str] = set()
    external: set[str] = set()

    for match in ART_REF_RE.finditer(text):
        ref = _normalize_reference_article(match.group(1))
        if not ref:
            continue

        w_start = max(0, match.start() - 64)
        w_end = min(len(text), match.end() + 96)
        window = text[w_start:w_end].lower()
        is_external = any(marker in window for marker in EXTERNAL_REFERENCE_MARKERS)

        if is_external:
            label = "Art. " + ref
            if "c.p." in window or "codice penale" in window:
                label += " c.p."
            elif "c.c." in window or "codice civile" in window:
                label += " c.c."
            elif "c.p.p" in window or "codice di procedura penale" in window:
                label += " c.p.p."
            elif "c.p.c" in window or "codice di procedura civile" in window:
                label += " c.p.c."
            elif "cost" in window:
                label += " Cost."
            external.add(label)
            continue

        if valid_internal_refs is not None and ref not in valid_internal_refs:
            external.add("Art. " + ref)
            continue

        internal.add(ref)

    return sorted(internal, key=_article_sort_key), sorted(external)


def extract_codice_amministrativo() -> tuple[list[AdministrativeArticleEntry], int]:
    opener = _build_opener()
    root_html = opener.open(BASE_NORMATTIVA_URL, timeout=60).read().decode("utf-8", "ignore")

    if SAVE_RAW_HTML:
        RAW_DEBUG_DIR.mkdir(parents=True, exist_ok=True)
        (RAW_DEBUG_DIR / "root.html").write_text(root_html, encoding="utf-8")

    article_paths: dict[str, str] = {}
    for m in ARTICLE_LINK_RE.finditer(root_html):
        ajax_path = html.unescape(m.group(1))
        label = _normalize_article_label(m.group(2))
        if not re.match(r"^\d+[a-z\-]*$", label):
            continue
        if label not in article_paths:
            article_paths[label] = ajax_path

    valid_internal_refs = set(article_paths.keys())

    rows: list[AdministrativeArticleEntry] = []
    for idx, (numero, ajax_path) in enumerate(sorted(article_paths.items(), key=lambda x: _article_sort_key(x[0])), start=1):
        req = urllib.request.Request(
            "https://www.normattiva.it" + ajax_path,
            headers={
                "User-Agent": "Mozilla/5.0",
                "X-Requested-With": "XMLHttpRequest",
                "Referer": BASE_NORMATTIVA_URL,
                "Accept": "*/*",
            },
        )
        payload = opener.open(req, timeout=60).read().decode("utf-8", "ignore")

        if SAVE_RAW_HTML and idx <= 20:
            (RAW_DEBUG_DIR / f"article_{idx:04d}_{numero}.html").write_text(payload, encoding="utf-8")

        start = payload.find('<div class="bodyTesto">')
        end = payload.find('<div class="d-flex justify-content-between', start)
        if start == -1 or end == -1:
            continue

        body_fragment = payload[start:end]
        lines = _html_to_lines(body_fragment)
        titolo, contenuto = _extract_title_and_body(lines, numero)

        if not contenuto and len(lines) > 1:
            contenuto = re.sub(r"\s+", " ", " ".join(lines[1:])).strip()

        internal_refs, external_refs = _extract_references(contenuto, valid_internal_refs=valid_internal_refs)

        rows.append(
            AdministrativeArticleEntry(
                numero=numero,
                titolo=titolo,
                contenuto=contenuto,
                reference=json.dumps(internal_refs, ensure_ascii=False),
                external_reference=json.dumps(external_refs, ensure_ascii=False),
            )
        )

    return rows, len(article_paths)


def save_csv(rows: list[AdministrativeArticleEntry], path: Path = OUTPUT_CSV_PATH) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["numero", "titolo", "contenuto", "reference", "external_reference"],
        )
        writer.writeheader()
        sorted_rows = sorted(rows, key=lambda r: _article_sort_key(r.numero))
        for row in sorted_rows:
            writer.writerow(
                {
                    "numero": row.numero,
                    "titolo": row.titolo,
                    "contenuto": row.contenuto,
                    "reference": row.reference,
                    "external_reference": row.external_reference,
                }
            )


rows, normattiva_unique_articles = extract_codice_amministrativo()
save_csv(rows)

print(f"Estratti articoli: {len(rows)}")
print(f"Articoli unici in albero Normattiva: {normattiva_unique_articles}")
print(f"CSV salvato in: {OUTPUT_CSV_PATH.resolve()}")
print(f"Timestamp UTC: {datetime.now(timezone.utc).isoformat()}")


Estratti articoli: 51
Articoli unici in albero Normattiva: 51
CSV salvato in: /Users/l.catello/Library/Mobile Documents/com~apple~CloudDocs/Magistrale Ingegneria Informatica/Tesi/LexCausa/src/data/statutes/codice_amministrativo_normattiva.csv
Timestamp UTC: 2026-02-20T21:55:47.793446+00:00


In [2]:
rows, normattiva_unique_articles = extract_codice_amministrativo()
save_csv(rows)

print(f"Estratti articoli: {len(rows)}")
print(f"Articoli unici in albero Normattiva: {normattiva_unique_articles}")
print(f"CSV scritto in: {OUTPUT_CSV_PATH.resolve()}")
print(f"Timestamp UTC: {datetime.now(timezone.utc).isoformat()}")

# Validazione rapida
nums = [r.numero for r in rows]
missing_title = sum(1 for r in rows if not (r.titolo or "").strip())
missing_text = sum(1 for r in rows if not (r.contenuto or "").strip())
print(f"Numero articoli distinti: {len(set(nums))}")
print(f"Titoli vuoti: {missing_title}")
print(f"Contenuti vuoti: {missing_text}")
print("Copertura albero Normattiva:", f"{len(rows)}/{normattiva_unique_articles}")


Estratti articoli: 51
Articoli unici in albero Normattiva: 51
CSV scritto in: /Users/l.catello/Library/Mobile Documents/com~apple~CloudDocs/Magistrale Ingegneria Informatica/Tesi/LexCausa/src/data/statutes/codice_amministrativo_normattiva.csv
Timestamp UTC: 2026-02-20T21:55:53.657441+00:00
Numero articoli distinti: 51
Titoli vuoti: 0
Contenuti vuoti: 0
Copertura albero Normattiva: 51/51
